# Montando o Drive e conectando ao modelo

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os

caminho_modelo = "/content/drive/MyDrive/techchallenge_fase3/biomistral-medquad-lora"

if os.path.exists(caminho_modelo):
    arquivos = os.listdir(caminho_modelo)
    print(f"✅ Modelo encontrado! {len(arquivos)} arquivos:")
    for arq in arquivos:
        print(f"   - {arq}")
else:
    print(" Modelo NÃO encontrado em:", caminho_modelo)


✅ Modelo encontrado! 7 arquivos:
   - tokenizer.model
   - README.md
   - adapter_model.safetensors
   - adapter_config.json
   - chat_template.jinja
   - tokenizer_config.json
   - tokenizer.json


In [3]:
! cd /content/Techchalleng3
! git pull origin main

fatal: not a git repository (or any of the parent directories): .git


## Instalar dependências

In [4]:
!pip install -q "datasets<3.0"

!python scripts/setup_data_colab.py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.8/796.8 kB 6.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 0.24.0 requires datasets>=3.0.0, but you have datasets 2.21.0 which is incompatible.
unsloth-zoo 2026.9.2 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 2.21.0 which is incompatible.
hf-gradio 0.4.1 requires gradio-client<3.0,>=2.0, but you have gradio-client 1.3.0 which is incompatible.
python3: can't open file '/content/scripts/setup_data_colab.py': [Errno 2] No such file or directory


In [5]:
# Força uninstall da versão 2.x e instala a 1.x
!pip uninstall -y numpy
!pip install -q "numpy==1.26.4"

# Verifica
import numpy as np
print(f"numpy {np.__version__}")
print(f"np.float_ disponível: {hasattr(np, 'float_')}")

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 61.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 0.24.0 requires datasets>=3.0.0, but you have datasets 2.21.0 which is incompatible.
unsloth-zoo 2026.9.2 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 2.21.0 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tifffile 2026.8.23 requires numpy>=2.1, b

In [6]:
# 1. Desinstala TODOS os pydantic
!pip uninstall -y pydantic pydantic-settings pydantic-core 2>&1 | tail -3

# 2. Limpa cache
!pip cache purge 2>&1 | tail -2

# 3. Instala pydantic v2 (compatível com gradio e ambiente atual)
!pip install -q "pydantic>=2.0,<3.0" "pydantic-settings"

# 4. Verifica TODOS os locais onde pydantic pode estar
import subprocess
result = subprocess.run(
    ["find", "/usr/local/lib", "/usr/lib", "-name", "pydantic", "-type", "d"],
    capture_output=True, text=True
)
print("📂 Localizações do pydantic:")
print(result.stdout)

# 5. Testa
import pydantic
import pydantic_core
print(f"\n✅ pydantic {pydantic.__version__}")
print(f"✅ pydantic_core {pydantic_core.__version__}")
print(f"✅ AliasChoices disponível: {hasattr(pydantic, 'AliasChoices') or hasattr(pydantic_core, 'AliasChoices')}")

# 6. Testa unsloth
try:
    from unsloth import FastLanguageModel
    print("✅ Unsloth importado com sucesso!")
except ImportError as e:
    print(f"❌ Unsloth falhou: {e}")

Found existing installation: pydantic_core 2.46.5
Uninstalling pydantic_core-2.46.5:
  Successfully uninstalled pydantic_core-2.46.5
Files removed: 109
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.6/472.6 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-sdk 0.4.3 requires websockets<17,>=14, but you have websockets 12.0 which is incompatible.
google-genai 2.12.1 requires websockets<17.0,>=13.0.0, but you have websockets 12.0 which is incompatible.
langsmith 0.11.1 requires websockets>=15.0, but you have websockets 12.0 which is incompatible.
google-adk 2.7.1 requires fastapi<1,>=0.133, but you

In [7]:
# Limpa conflitos de versão


!pip install -q "chromadb==0.4.18"
!pip install -q "gradio==4.44.0"
!pip install -q "huggingface_hub" "transformers"

# Unsloth (LLM)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q "bitsandbytes" "accelerate" "trl" "peft"

# Patches importantes
!pip install -q "reportlab"

# Verifica
import torch
print(f"✅ PyTorch {torch.__version__}")
print(f"✅ CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.6/224.6 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 1.3.0 requires websockets<13.0,>=10.0, but you have websockets 17.1 which is incompatible.
hf-gradio 0.4.1 requires gradio-client<3.0,>=2.0, but you have gradio-client 1.3.0 which is incompatible.
langgraph-sdk 0.4.3 requires websockets<17,>=14, but you have websockets 17.1 which is incompatible.
google-genai 2.12.1 requires websockets<17.0,>=13.0.0, but you have websockets 17.1 which is incompatible.
google-adk 2.7.1 requires fastapi<1,>=0.133, but you have fastapi 0.125.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is i

## Clonar repositório

## Passo 5

In [ ]:
import os
from pathlib import Path

# Limpa qualquer clone anterior
if Path("Techchalleng3").exists():
    !rm -rf Techchalleng3

# Clona o repo
!git clone https://github.com/Flamers-Team/Techchalleng3.git

# Entra na pasta
%cd Techchalleng3

# Confirma
print(f"📁 Estamos em: {os.getcwd()}")
print(f"📂 Arquivos: {os.listdir('.')[:10]}")

Cloning into 'Techchalleng3'...
remote: Enumerating objects: 11617, done.
remote: Counting objects: 100% (11617/11617), done.
remote: Compressing objects: 100% (4715/4715), done.
remote: Total 11617 (delta 6953), reused 11527 (delta 6873), pack-reused 0 (from 0)
Receiving objects: 100% (11617/11617), 10.76 MiB | 21.39 MiB/s, done.
Resolving deltas: 100% (6953/6953), done.


## Passo 6

In [ ]:
import shutil
from pathlib import Path

DRIVE_MODEL = Path("/content/drive/MyDrive/techchallenge_fase3/biomistral-medquad-lora")
LOCAL_MODEL = Path("/content/biomistral-medquad-lora")

if not LOCAL_MODEL.exists() and DRIVE_MODEL.exists():
    print(f"📦 Copiando modelo do Drive...")
    shutil.copytree(DRIVE_MODEL, LOCAL_MODEL)
    print(f"✅ Copiado! {len(list(LOCAL_MODEL.iterdir()))} arquivos")
elif LOCAL_MODEL.exists():
    print(f"✅ Modelo já está em {LOCAL_MODEL}")
else:
    print(f"❌ Drive não tem o modelo em {DRIVE_MODEL}")

## Passo 7

In [ ]:
import gradio_client
import os

gc_path = os.path.join(os.path.dirname(gradio_client.__file__), "utils.py")

with open(gc_path, "r") as f:
    content = f.read()

# Patch TODAS as ocorrências (não só 'const')
patches = [
    ('    if "enum" in schema:', '    if isinstance(schema, dict) and "enum" in schema:'),
    ('    if "const" in schema:', '    if isinstance(schema, dict) and "const" in schema:'),
    ('if "enum" in schema:', 'if isinstance(schema, dict) and "enum" in schema:'),
    ('if "const" in schema:', 'if isinstance(schema, dict) and "const" in schema:'),
]

patched = 0
for bug, fix in patches:
    if bug in content and fix not in content:
        content = content.replace(bug, fix)
        patched += 1

if patched > 0:
    with open(gc_path, "w") as f:
        f.write(content)
    print(f"✅ {patched} patches aplicados no gradio_client")
else:
    print("ℹ️  Patch já estava aplicado")


## Passo 8

In [ ]:
import os
os.chdir("/content/Techchalleng3")

with open("src/ui/gradio_app.py", "r") as f:
    code = f.read()

if "share=False" in code:
    code = code.replace("share=False", "share=True")
    with open("src/ui/gradio_app.py", "w") as f:
        f.write(code)
    print("✅ share=False → share=True")
else:
    print("ℹ️  share já estava True")

## Passo 9

In [ ]:
# Copia modelo pro caminho que a UI espera (/content/Techchalleng3/biomistral-medquad-lora)
import shutil
from pathlib import Path

SOURCE = Path("/content/biomistral-medquad-lora")
TARGET = Path("/content/Techchalleng3/biomistral-medquad-lora")

if not TARGET.exists() and SOURCE.exists():
    print("📦 Copiando modelo pro caminho que a UI espera...")
    shutil.copytree(SOURCE, TARGET)
    print(f"✅ Copiado! {len(list(TARGET.iterdir()))} arquivos")
elif TARGET.exists():
    print(f"✅ Modelo já está em {TARGET}")
else:
    print(f"❌ Modelo não encontrado em {SOURCE}")


## ChatBulário COMPLETO

In [ ]:
!python src/rag/build_index_chatbulario.py

##ChatBulário menor para DEMO

In [ ]:
# (Célula removida - já indexamos ChatBulário no Passo 9)
print("ℹ️  ChatBulário já indexado anteriormente")


### CORREÇÃO DE ERRO no chromadb

In [ ]:
# Substitui np.float_ por np.float64 diretamente no código do chromadb
import subprocess
result = subprocess.run(
    ["grep", "-rl", "np.float_", "/usr/local/lib/python3.13/dist-packages/chromadb/"],
    capture_output=True, text=True
)
files_to_patch = result.stdout.strip().split("\n")
print(f"Encontrado em {len(files_to_patch)} arquivos")

for f in files_to_patch:
    if f and f.endswith(".py"):
        with open(f, "r") as fp:
            content = fp.read()
        content = content.replace("np.float_", "np.float64")
        with open(f, "w") as fp:
            fp.write(content)
        print(f"✅ Patch: {f}")

# Verifica
import chromadb
print(f"✅ chromadb {chromadb.__version__}")

# RAG

In [ ]:
import os
import time
os.chdir("/content/Techchalleng3")

t0 = time.time()

# 1. RAG
print("=" * 70)
print("📚 [1/2] Carregando RAG (ChromaDB)...")
print("=" * 70)

from src.rag.retriever import Retriever

retriever = Retriever()
print()

# 2. LLM
print("=" * 70)
print("🤖 [2/2] Carregando LLM (BioMistral + LoRA)...")
print("=" * 70)
from src.llm.client import LLMClient
llm = LLMClient(lora_path=Path("/content/biomistral-medquad-lora"))

print(f"\n⏱️  Tempo total: {time.time() - t0:.1f}s")

if llm.use_mock:
    print("\n⚠️  LLM EM MODO MOCK — verifique se o modelo foi carregado!")
else:
    print("\n✅ LLM REAL carregado — UI vai usar a IA treinada!")

## Passo 10 - Subir UI

⚠️ **Importante**: Antes de subir a UI, garante que:
1. O patch do gradio_client foi aplicado (célula anterior)
2. O modelo foi copiado para `/content/Techchalleng3/biomistral-medquad-lora`
3. O RAG foi carregado (Passo 9)

Se aparecer `Internal Server Error`, é bug do gradio_client — patcha de novo.


In [ ]:
# Downgrade huggingface_hub pra versão que ainda tem HfFolder
!pip install -q "huggingface_hub==0.20.0"

# Verifica
import huggingface_hub
print(f"✅ huggingface_hub {huggingface_hub.__version__}")
print(f"✅ HfFolder disponível: {hasattr(huggingface_hub, 'HfFolder')}")

In [ ]:
import os
os.chdir("/content/Techchalleng3")

print("=" * 70)
print("🚀 SUBINDO UI GRADIO...")
print("=" * 70)
print("Aguarde 30s. Vai aparecer uma URL pública.")
print()

!python src/ui/gradio_app.py